In [ ]:
# Import libraries
from delta import DeltaTable
from pyspark.sql.functions import col, when, first, max, lit, concat, create_map, lower
import json
from itertools import chain

In [ ]:
# Define source schema and tables for fact_demand table in gold
source_schema = 'dbo'
source_tables = ['market']

In [ ]:
# Define parameters with None, as values will come from pipeline activities
# Source
source_lakehouse_path = None 

# Target
target_lakehouse_path = None 
target_schema = None
target_table = None

In [ ]:
# Define variables using values from parameters
source_paths = [f'{source_lakehouse_path}/Tables/{source_schema}/{table}' for table in source_tables]
target_path = f'{target_lakehouse_path}/Tables/{target_schema}/{target_table}'

# Load Helper Notebooks

Load helper notebook with the schema, merge keys and indicator mapping definition of gold tables inside "gold_objects" dictionary: 


In [ ]:
%run ./NB_Gold_Table_Schemas

In [ ]:
# Use gold_objects dictionary to define aditional variables
target_table_schema = gold_objects[f"{target_table}"]["schema"]

# Read and Transform New Data

In [ ]:
# Create a list of dataframes with data of tables in source_paths list. Only read unique values of relevant columns 
dfs = [
    spark.read
        .format("delta")
        .load(path)
        .select(
            col("geo_id"),
            col("geo_name")
        )
        .dropDuplicates()
    for path in source_paths
]

In [ ]:
# Use the first dataframe in dfs as df, then iterate over the rest of the dataframes in the list performing an union between them
df = dfs[0]

for source_df in dfs[1:]:
    df = df.unionByName(source_df)

# Remove duplicates across all sources
df = df.dropDuplicates(["geo_id", "geo_name"])

In [ ]:
# Create a dictionary
country_codes = {
    "Germany": "DE",
    "Belgium": "BE",
    "Netherlands": "NL",
    "Portugal": "PT",
    "France": "FR",
    "Spain": "ES"
}

# Convert each item to map type
mapping_expr = create_map([lit(x) for x in chain(*country_codes.items())])

# Add country code and corresponding flag URL
df = (
    df
    .withColumn("country_code", mapping_expr[col("geo_name")])
    .withColumn("flag", concat(lit("https://flagcdn.com/"), lower(col("country_code")), lit(".svg")))
)

# Update Table

In [ ]:
# Full load
# Create the table if it doesnt exists
try:
    delta_table = DeltaTable.forPath(spark, target_path)
    print('Table already exists')
except:
    delta_table = DeltaTable.createIfNotExists(spark) \
    .location(target_path) \
    .addColumns(target_table_schema) \
    .execute()
    print('Table has been created')

# Recreate the table without infering schema
df.write.format("delta").mode("overwrite").save(target_path) 

# Statistics
rows_inserted = df.count()
rows_updated = 0

In [ ]:
# File a dictionary with operational statistics
result = {
    "maxdate": None,
    "rows_inserted": rows_inserted,
    "rows_updated": rows_updated
}

# Make the result values queriable for other activities in the pipeline
mssparkutils.notebook.exit(json.dumps(result))